<a href="https://colab.research.google.com/github/ngyxxgwxi/Tiktok-TechJam/blob/main/Tiktok_TechJam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install rank-bm25 faiss-cpu sentence-transformers

In [15]:
import numpy as np
from typing import Dict, List, Tuple
from rank_bm25 import BM25Okapi
import faiss
from sentence_transformers import SentenceTransformer

class HybridSearchEngine:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.encoder = SentenceTransformer(model_name)
        self.doc_ids: List[str] = []
        self.documents: List[Dict] = []
        self.bm25_index: BM25Okapi = None
        self.faiss_index: faiss.IndexFlatIP = None

    def _tokenize(self, text: str) -> List[str]:
        return text.lower().split()

    def build_index(self, catalog: List[Dict]):
        self.documents = catalog
        self.doc_ids = [item.get("asin", str(i)) for i, item in enumerate(catalog)]

        corpus_texts = [
            f"{item.get('title', '')} {item.get('description', '')}"
            for item in catalog
        ]


        tokenized_cus = [self._tokenize(doc) for doc in corpus_texts]
        self.bm25_index = BM25Okapi(tokenized_corpus)

        embeddings = self.encoder.encode(
            corpus_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True
        )
        dimension = embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dimension)
        self.faiss_index.add(np.array(embeddings, dtype=np.float32))

    def _retrieve_bm25(self, query: str, top_k: int) -> List[Tuple[str, float]]:
        tokenized_query = self._tokenize(query)
        scores = self.bm25_index.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.doc_ids[idx], float(scores[idx])) for idx in top_indices]

    def _retrieve_dense(self, query: str, top_k: int) -> List[Tuple[str, float]]:
        query_vector = self.encoder.encode([query], normalize_embeddings=True)
        scores, indices = self.faiss_index.search(
            np.array(query_vector, dtype=np.float32), top_k
        )
        return [
            (self.doc_ids[idx], float(score))
            for idx, score in zip(indices[0], scores[0])
            if idx != -1
        ]

    def reciprocal_rank_fusion(
        self,
        bm25_results: List[Tuple[str, float]],
        dense_results: List[Tuple[str, float]],
        k_constant: int = 60,
        weights: Tuple[float, float] = (1.0, 1.0)
    ) -> List[Tuple[str, float]]:
        rrf_scores: Dict[str, float] = {}
        bm25_weight, dense_weight = weights

        for rank, (doc_id, _) in enumerate(bm25_results, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + bm25_weight * (1.0 / (k_constant + rank))

        for rank, (doc_id, _) in enumerate(dense_results, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + dense_weight * (1.0 / (k_constant + rank))

        return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    def search(
        self,
        query: str,
        candidate_k: int = 100,
        final_top_k: int = 10,
        track: str = "browsing"
    ) -> List[Dict]:
        weights = (1.5, 0.5) if track == "buying" else (1.0, 1.0)
        bm25_candidates = self._retrieve_bm25(query, top_k=candidate_k)
        dense_candidates = self._retrieve_dense(query, top_k=candidate_k)

        fused_rankings = self.reciprocal_rank_fusion(
            bm25_candidates, dense_candidates, k_constant=60, weights=weights
        )

        doc_map = {doc.get("asin", str(i)): doc for i, doc in enumerate(self.documents)}
        return [
            {**doc_map[doc_id], "rrf_score": score}
            for doc_id, score in fused_rankings[:final_top_k]
            if doc_id in doc_map
        ]


In [24]:
import numpy as np
from typing import Dict, List, Tuple
from rank_bm25 import BM25Okapi
import faiss
from sentence_transformers import SentenceTransformer

class HybridSearchEngine:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.encoder = SentenceTransformer(model_name)
        self.doc_ids: List[str] = []
        self.documents: List[Dict] = []
        self.bm25_index: BM25Okapi = None
        self.faiss_index: faiss.IndexFlatIP = None

    def _tokenize(self, text: str) -> List[str]:
        return text.lower().split()

    def build_index(self, catalog: List[Dict]):
        self.documents = catalog
        self.doc_ids = [str(item.get("parent_asin", item.get("id", i))) for i, item in enumerate(catalog)]

        corpus_texts = []
        for item in catalog:
            title = str(item.get("title", ""))

            features = item.get("features", [])
            features_text = " ".join(features) if isinstance(features, list) else str(features)

            desc = item.get("description", [])
            desc_text = " ".join(desc) if isinstance(desc, list) else str(desc)

            cats = item.get("categories", [])
            cats_text = " ".join(cats) if isinstance(cats, list) else str(cats)

            full_text = f"{title} {cats_text} {features_text} {desc_text}".strip()
            corpus_texts.append(full_text)

        tokenized_corpus = [self._tokenize(doc) for doc in corpus_texts]
        self.bm25_index = BM25Okapi(tokenized_corpus)

        embeddings = self.encoder.encode(
            corpus_texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True
        )
        dimension = embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dimension)
        self.faiss_index.add(np.array(embeddings, dtype=np.float32))

    def _retrieve_bm25(self, query: str, top_k: int) -> List[Tuple[str, float]]:
        tokenized_query = self._tokenize(query)
        scores = self.bm25_index.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.doc_ids[idx], float(scores[idx])) for idx in top_indices]

    def _retrieve_dense(self, query: str, top_k: int) -> List[Tuple[str, float]]:
        query_vector = self.encoder.encode([query], normalize_embeddings=True)
        scores, indices = self.faiss_index.search(
            np.array(query_vector, dtype=np.float32), top_k
        )
        return [
            (self.doc_ids[idx], float(score))
            for idx, score in zip(indices[0], scores[0])
            if idx != -1
        ]

    def reciprocal_rank_fusion(
        self,
        bm25_results: List[Tuple[str, float]],
        dense_results: List[Tuple[str, float]],
        k_constant: int = 60,
        weights: Tuple[float, float] = (1.0, 1.0)
    ) -> List[Tuple[str, float]]:
        rrf_scores: Dict[str, float] = {}
        bm25_weight, dense_weight = weights

        for rank, (doc_id, _) in enumerate(bm25_results, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + bm25_weight * (1.0 / (k_constant + rank))

        for rank, (doc_id, _) in enumerate(dense_results, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + dense_weight * (1.0 / (k_constant + rank))

        return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    def search(
        self,
        query: str,
        candidate_k: int = 100,
        final_top_k: int = 10,
        track: str = "browsing"
    ) -> List[Dict]:
        weights = (1.5, 0.5) if track == "buying" else (1.0, 1.0)
        bm25_candidates = self._retrieve_bm25(query, top_k=candidate_k)
        dense_candidates = self._retrieve_dense(query, top_k=candidate_k)

        fused_rankings = self.reciprocal_rank_fusion(
            bm25_candidates, dense_candidates, k_constant=60, weights=weights
        )

        doc_map = {str(item.get("parent_asin", item.get("id", i))): item for i, item in enumerate(self.documents)}
        return [
            {**doc_map[doc_id], "rrf_score": score}
            for doc_id, score in fused_rankings[:final_top_k]
            if doc_id in doc_map
        ]

In [19]:
!pip install -q rank-bm25 faiss-cpu sentence-transformers

In [23]:
import json
import os
from typing import Dict, List, Tuple
import numpy as np
from rank_bm25 import BM25Okapi
import faiss
from sentence_transformers import SentenceTransformer

FILE_PATH = "/content/sample_data/catalog.jsonl"

catalog = []
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"Could not find file at '{FILE_PATH}'. Check your file name in the Colab sidebar!")

print(f"Loading catalog from {FILE_PATH}...")
with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            catalog.append(json.loads(line))

print(f"Successfully loaded {len(catalog):,} catalog items!")

class HybridSearchEngine:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print("Initializing Sentence Transformer encoder...")
        self.encoder = SentenceTransformer(model_name)
        self.doc_ids: List[str] = []
        self.documents: List[Dict] = []
        self.bm25_index: BM25Okapi = None
        self.faiss_index: faiss.IndexFlatIP = None

    def _tokenize(self, text: str) -> List[str]:
        return text.lower().split()

    def build_index(self, catalog: List[Dict]):
        self.documents = catalog
        self.doc_ids = [str(item.get("asin", item.get("id", i))) for i, item in enumerate(catalog)]

        corpus_texts = []
        for item in catalog:
            title = str(item.get("title", ""))
            desc = str(item.get("description", ""))
            if isinstance(desc, list):
                desc = " ".join(desc)
            corpus_texts.append(f"{title} {desc}".strip())

        print("Building BM25 sparse index...")
        tokenized_corpus = [self._tokenize(doc) for doc in corpus_texts]
        self.bm25_index = BM25Okapi(tokenized_corpus)

        print("Building FAISS dense vector index...")
        embeddings = self.encoder.encode(
            corpus_texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True
        )
        dimension = embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dimension)
        self.faiss_index.add(np.array(embeddings, dtype=np.float32))
        print("Indexing complete!")

    def _retrieve_bm25(self, query: str, top_k: int) -> List[Tuple[str, float]]:
        tokenized_query = self._tokenize(query)
        scores = self.bm25_index.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.doc_ids[idx], float(scores[idx])) for idx in top_indices]

    def _retrieve_dense(self, query: str, top_k: int) -> List[Tuple[str, float]]:
        query_vector = self.encoder.encode([query], normalize_embeddings=True)
        scores, indices = self.faiss_index.search(np.array(query_vector, dtype=np.float32), top_k)
        return [(self.doc_ids[idx], float(score)) for idx, score in zip(indices[0], scores[0]) if idx != -1]

    def reciprocal_rank_fusion(
        self,
        bm25_results: List[Tuple[str, float]],
        dense_results: List[Tuple[str, float]],
        k_constant: int = 60,
        weights: Tuple[float, float] = (1.0, 1.0)
    ) -> List[Tuple[str, float]]:
        rrf_scores: Dict[str, float] = {}
        bm25_weight, dense_weight = weights

        for rank, (doc_id, _) in enumerate(bm25_results, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + bm25_weight * (1.0 / (k_constant + rank))

        for rank, (doc_id, _) in enumerate(dense_results, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + dense_weight * (1.0 / (k_constant + rank))

        return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    def search(
        self, query: str, candidate_k: int = 100, final_top_k: int = 10, track: str = "browsing"
    ) -> List[Dict]:
        weights = (1.5, 0.5) if track == "buying" else (1.0, 1.0)
        bm25_candidates = self._retrieve_bm25(query, top_k=candidate_k)
        dense_candidates = self._retrieve_dense(query, top_k=candidate_k)

        fused_rankings = self.reciprocal_rank_fusion(
            bm25_candidates, dense_candidates, k_constant=60, weights=weights
        )

        doc_map = {str(item.get("asin", item.get("id", i))): item for i, item in enumerate(self.documents)}
        return [
            {**doc_map[doc_id], "rrf_score": score}
            for doc_id, score in fused_rankings[:final_top_k]
            if doc_id in doc_map
        ]


engine = HybridSearchEngine()
engine.build_index(catalog)

results = engine.search(query="running shoes", track="buying", final_top_k=5)
print(f"\nFound {len(results)} matches!")

for i, res in enumerate(results, start=1):
    title = res.get("title", "No Title")
    asin = res.get("asin", "N/A")
    score = res["rrf_score"]
    print(f"{i}. [{asin}] {title[:60]}... | Score: {score:.4f}")

Loading catalog from /content/sample_data/catalog.jsonl...
Successfully loaded 50,000 catalog items!
Initializing Sentence Transformer encoder...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Building BM25 sparse index...
Building FAISS dense vector index...


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

Indexing complete!

Found 5 matches!
1. [N/A] Running Shoes Athletic Shoes Slip-On Sport Shoes Lightweight... | Score: 0.0313
2. [N/A] Women's Trial Running Shoes Men's Walking Shoes Minimalist B... | Score: 0.0291
3. [N/A] Air Balance Men's Black Lightweight Running Shoes... | Score: 0.0286
4. [N/A] Air Balance Women's White/Black/Fuchsia Lightweight Running ... | Score: 0.0269
5. [N/A] Nike mens Running Shoes... | Score: 0.0263


In [21]:
print("Catalog Keys:", catalog[0].keys())
print("First item sample:", catalog[0])

Catalog Keys: dict_keys(['parent_asin', 'title', 'features', 'description', 'price', 'categories', 'details', 'average_rating', 'rating_number', 'store'])
First item sample: {'parent_asin': 'B07K34RX5J', 'title': 'Kandinsky Statement Earrings for Women by Spirit Hoops, Fabric, Lightweight Drop and Dangle Stainless Steel Hoop Earrings for Women Fashion, Artsy', 'features': ['Spandex', 'Made in USA and Imported', 'Fashion jewelry: Beautiful fabric earrings for women featuring a painting by Kandinsky are wearable art.', 'COMFORTABLE: Lightweight dangle earrings, hoops measure approximately 2 inches in diameter. Drop and Dangle hoops with soft fabric covers handmade with care in the U.S.A!', 'Great for women of all ages.', 'GREAT QUALITY: Earrings are made with high-quality, hypoallergenic stainless steel.', 'Comes in a jewelry gift box for gift giving and safekeeping.'], 'description': ['Kandinsky earrings by Spirit Hoops have a unique, romantic look and are sure to turn heads. Art is dis

In [25]:
import json
import os
from typing import Dict, List, Tuple
import numpy as np
from rank_bm25 import BM25Okapi
import faiss
from sentence_transformers import SentenceTransformer

FILE_PATH = "/content/sample_data/catalog.jsonl"

catalog = []
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"❌ Could not find file at '{FILE_PATH}'.")

print(f"Loading catalog from {FILE_PATH}...")
with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            catalog.append(json.loads(line))

print(f"Successfully loaded {len(catalog):,} catalog items!")

class HybridSearchEngine:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print("Initializing Sentence Transformer encoder...")
        self.encoder = SentenceTransformer(model_name)
        self.doc_ids: List[str] = []
        self.documents: List[Dict] = []
        self.bm25_index: BM25Okapi = None
        self.faiss_index: faiss.IndexFlatIP = None

    def _tokenize(self, text: str) -> List[str]:
        return text.lower().split()

    def build_index(self, catalog: List[Dict]):
        self.documents = catalog
        self.doc_ids = [str(item.get("parent_asin", i)) for i, item in enumerate(catalog)]

        corpus_texts = []
        for item in catalog:
            title = str(item.get("title", ""))

            features = item.get("features", [])
            features_text = " ".join(features) if isinstance(features, list) else str(features)

            desc = item.get("description", [])
            desc_text = " ".join(desc) if isinstance(desc, list) else str(desc)

            cats = item.get("categories", [])
            cats_text = " ".join(cats) if isinstance(cats, list) else str(cats)

            full_text = f"{title} {cats_text} {features_text} {desc_text}".strip()
            corpus_texts.append(full_text)

        print("Building BM25 sparse index...")
        tokenized_corpus = [self._tokenize(doc) for doc in corpus_texts]
        self.bm25_index = BM25Okapi(tokenized_corpus)

        print("Building FAISS dense vector index...")
        embeddings = self.encoder.encode(
            corpus_texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True
        )
        dimension = embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dimension)
        self.faiss_index.add(np.array(embeddings, dtype=np.float32))
        print("Indexing complete!")

    def _retrieve_bm25(self, query: str, top_k: int) -> List[Tuple[str, float]]:
        tokenized_query = self._tokenize(query)
        scores = self.bm25_index.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.doc_ids[idx], float(scores[idx])) for idx in top_indices]

    def _retrieve_dense(self, query: str, top_k: int) -> List[Tuple[str, float]]:
        query_vector = self.encoder.encode([query], normalize_embeddings=True)
        scores, indices = self.faiss_index.search(np.array(query_vector, dtype=np.float32), top_k)
        return [(self.doc_ids[idx], float(score)) for idx, score in zip(indices[0], scores[0]) if idx != -1]

    def reciprocal_rank_fusion(
        self,
        bm25_results: List[Tuple[str, float]],
        dense_results: List[Tuple[str, float]],
        k_constant: int = 60,
        weights: Tuple[float, float] = (1.0, 1.0)
    ) -> List[Tuple[str, float]]:
        rrf_scores: Dict[str, float] = {}
        bm25_weight, dense_weight = weights

        for rank, (doc_id, _) in enumerate(bm25_results, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + bm25_weight * (1.0 / (k_constant + rank))

        for rank, (doc_id, _) in enumerate(dense_results, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + dense_weight * (1.0 / (k_constant + rank))

        return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    def search(
        self, query: str, candidate_k: int = 100, final_top_k: int = 10, track: str = "browsing"
    ) -> List[Dict]:
        weights = (1.5, 0.5) if track == "buying" else (1.0, 1.0)
        bm25_candidates = self._retrieve_bm25(query, top_k=candidate_k)
        dense_candidates = self._retrieve_dense(query, top_k=candidate_k)

        fused_rankings = self.reciprocal_rank_fusion(
            bm25_candidates, dense_candidates, k_constant=60, weights=weights
        )

        doc_map = {str(item.get("parent_asin", i)): item for i, item in enumerate(self.documents)}
        return [
            {**doc_map[doc_id], "rrf_score": score}
            for doc_id, score in fused_rankings[:final_top_k]
            if doc_id in doc_map
        ]

engine = HybridSearchEngine()
engine.build_index(catalog)

results = engine.search(query="running shoes", track="buying", final_top_k=5)
print(f"\nFound {len(results)} matches!")

for i, res in enumerate(results, start=1):
    title = res.get("title", "No Title")
    asin = res.get("parent_asin", "N/A")
    score = res["rrf_score"]
    print(f"{i}. [{asin}] {title[:60]}... | Score: {score:.4f}")

Loading catalog from /content/sample_data/catalog.jsonl...
Successfully loaded 50,000 catalog items!
Initializing Sentence Transformer encoder...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Building BM25 sparse index...
Building FAISS dense vector index...


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

Indexing complete!

Found 5 matches!
1. [B09943N1V8] Nike mens Running Shoes... | Score: 0.0316
2. [B00K7L5DIG] Nike Women's Zoom WUNFLO Running Shoes... | Score: 0.0286
3. [B00BPDD47M] ASICS Men's Gel-Kayano 20 Running Shoes... | Score: 0.0285
4. [B0885ZH2TK] Sneakers for Men Athletic Tennis Walking Shoes Fashion Sport... | Score: 0.0277
5. [B07L3QGFFS] KULL FLOP Tennis Shoes for Women Wear-Resistant On Running S... | Score: 0.0273


In [26]:
import re
from typing import Dict, Any, List, Tuple

class DialogStateTracker:
    def __init__(self, candidate_cutoff_threshold: int = 20):
        self.reset()
        self.candidate_cutoff_threshold = candidate_cutoff_threshold

    def reset(self):
        """Reset session state for a new user turn/session."""
        self.history: List[Dict[str, str]] = []
        self.slots: Dict[str, Any] = {
            "gender": None,
            "brand": None,
            "category": None,
            "max_price": None,
            "color": None
        }
        self.current_intent: str = "browsing"

    def classify_intent(self, user_utterance: str) -> str:
        """
        Pillar I: Dual-Track Intent Classifier
        Detects 'buying' (high constraint/spec intent) vs 'browsing' (exploratory).
        """
        buying_triggers = [
            r"buy", r"price", r"under \$\d+", r"cost", r"size \d+",
            r"brand", r"looking for a specific", r"need exact", r"model"
        ]
        text_lower = user_utterance.lower()

        if any(re.search(pat, text_lower) for pat in buying_triggers) or self.slots["brand"] or self.slots["max_price"]:
            return "buying"
        return "browsing"

    def update_state(self, user_utterance: str) -> Dict[str, Any]:
        """
        Pillar II & III: Dynamic Context Programming
        Extracts slots, handles Intent Overrides (slot rewrites), and updates short-term context.
        """
        text = user_utterance.lower()

        if any(kw in text for kw in ["instead", "actually", "nevermind", "change to"]):
            if "brand" in text or any(b in text for b in ["nike", "adidas", "asics"]):
                self.slots["brand"] = None  # Clear brand slot for override

        if "women" in text or "female" in text or "womens" in text:
            self.slots["gender"] = "Women"
        elif "men" in text or "male" in text or "mens" in text:
            self.slots["gender"] = "Men"

        brands = ["nike", "adidas", "asics", "puma", "under armour", "new balance"]
        for b in brands:
            if b in text:
                self.slots["brand"] = b.title()

        price_match = re.search(r"under \$?(\d+)", text)
        if price_match:
            self.slots["max_price"] = float(price_match.group(1))

        self.current_intent = self.classify_intent(user_utterance)

        self.history.append({"role": "user", "content": user_utterance})
        return self.slots

    def construct_search_query(self, latest_utterance: str) -> str:
        """Builds an augmented context query from accumulated dialogue slots."""
        active_slots = [str(val) for key, val in self.slots.items() if val and key != "max_price"]
        augmented_query = " ".join(active_slots + [latest_utterance])
        return augmented_query.strip()

    def should_trigger_proactive_guidance(self, candidate_count: int) -> Tuple[bool, str]:
        """
        Pillar II: Proactive Guidance Cutoff
        Triggers structured clarification prompt if candidate pool overloaded (Over-Generality).
        """
        if candidate_count > self.candidate_cutoff_threshold and not self.slots["brand"]:
            clarification = "I found many matching options! Do you have a preferred brand (e.g., Nike, ASICS) or a target price range?"
            return True, clarification
        return False, ""

dst = DialogStateTracker(candidate_cutoff_threshold=10)

turns = [
    "Turn 1: Looking for comfortable running shoes",
    "Turn 2: Show me Nike shoes for men",
    "Turn 3: Actually, change that to ASICS instead"
]

print("--- Multi-Turn Conversation Simulation ---\n")
for turn_str in turns:
    utterance = turn_str.split(": ")[1]

    updated_slots = dst.update_state(utterance)
    query = dst.construct_search_query(utterance)

    results = engine.search(query=query, track=dst.current_intent, final_top_k=50)

    trigger_cutoff, prompt = dst.should_trigger_proactive_guidance(len(results))

    print(f"💬 User: '{utterance}'")
    print(f"   ├─ Detected Track: [{dst.current_intent.upper()}]")
    print(f"   ├─ Extracted Slots: {updated_slots}")
    print(f"   ├─ Augmented Query: '{query}'")
    if trigger_cutoff:
        print(f"   └─ ⚡ PROACTIVE CUTOFF TRIGGERED: \"{prompt}\"")
    else:
        print(f"   └─ 🎯 Top Match: [{results[0].get('parent_asin')}] {results[0].get('title')[:50]}...")
    print("-" * 65)

--- Multi-Turn Conversation Simulation ---

💬 User: 'Looking for comfortable running shoes'
   ├─ Detected Track: [BROWSING]
   ├─ Extracted Slots: {'gender': None, 'brand': None, 'category': None, 'max_price': None, 'color': None}
   ├─ Augmented Query: 'Looking for comfortable running shoes'
   └─ ⚡ PROACTIVE CUTOFF TRIGGERED: "I found many matching options! Do you have a preferred brand (e.g., Nike, ASICS) or a target price range?"
-----------------------------------------------------------------
💬 User: 'Show me Nike shoes for men'
   ├─ Detected Track: [BUYING]
   ├─ Extracted Slots: {'gender': 'Men', 'brand': 'Nike', 'category': None, 'max_price': None, 'color': None}
   ├─ Augmented Query: 'Men Nike Show me Nike shoes for men'
   └─ 🎯 Top Match: [B00A40GKZU] Nike Men's Air Max 90 Essential Running Shoes, Bla...
-----------------------------------------------------------------
💬 User: 'Actually, change that to ASICS instead'
   ├─ Detected Track: [BUYING]
   ├─ Extracted Slots: {

In [27]:
import re
from typing import Dict, Any, List, Tuple

class OptimizedDialogStateTracker(DialogStateTracker):
    def construct_search_query(self, latest_utterance: str) -> str:
        """
        Builds a clean, deduplicated context query from accumulated dialogue slots.
        Avoids repeating words like 'Men Nike Show me Nike shoes for men'.
        """
        active_slots = [str(val) for key, val in self.slots.items() if val and key != "max_price"]

        combined_text = " ".join(active_slots + [latest_utterance])
        tokens = combined_text.split()

        seen = set()
        deduped_tokens = []
        for token in tokens:
            lower_token = token.lower().strip(",.!?")
            if lower_token not in seen and lower_token not in ["actually", "change", "that", "to", "instead", "show", "me"]:
                seen.add(lower_token)
                deduped_tokens.append(token)

        return " ".join(deduped_tokens).strip()

class SemanticLLMReranker:
    """
    Reranks the top candidate pool from hybrid retrieval to maximize MRR.
    Uses precise slot matching + semantic alignment scoring.
    """
    def rerank(
        self,
        candidates: List[Dict],
        slots: Dict[str, Any],
        user_query: str,
        top_k: int = 10
    ) -> List[Dict]:
        scored_candidates = []

        target_brand = (slots.get("brand") or "").lower()
        target_gender = (slots.get("gender") or "").lower()
        max_price = slots.get("max_price")

        for item in candidates:
            score = item.get("rrf_score", 0.0)

            title = str(item.get("title", "")).lower()
            desc = " ".join(item.get("description", [])) if isinstance(item.get("description"), list) else str(item.get("description", "")).lower()
            item_text = f"{title} {desc}"

            if target_brand and target_brand not in item_text:
                score *= 0.1

            if target_gender:
                if target_gender == "men" and ("women" in title or "women's" in title):
                    score *= 0.2
                elif target_gender == "women" and ("men" in title and "women" not in title):
                    score *= 0.2

            price = item.get("price")
            if max_price and price and isinstance(price, (int, float)):
                if price > max_price:
                    score *= 0.05

            for word in user_query.lower().split():
                if len(word) > 3 and word in item_text:
                    score += 0.005

            scored_candidates.append({**item, "final_score": score})

        scored_candidates.sort(key=lambda x: x["final_score"], reverse=True)
        return scored_candidates[:top_k]

opt_dst = OptimizedDialogStateTracker(candidate_cutoff_threshold=10)
reranker = SemanticLLMReranker()

turns = [
    "Turn 1: Looking for comfortable running shoes",
    "Turn 2: Show me Nike shoes for men",
    "Turn 3: Actually, change that to ASICS instead"
]

print("=== COMPLETE PIPELINE MULTI-TURN TEST ===\n")
for turn_str in turns:
    utterance = turn_str.split(": ")[1]

    updated_slots = opt_dst.update_state(utterance)
    clean_query = opt_dst.construct_search_query(utterance)

    raw_candidates = engine.search(query=clean_query, track=opt_dst.current_intent, candidate_k=100, final_top_k=50)

    trigger_cutoff, prompt = opt_dst.should_trigger_proactive_guidance(len(raw_candidates))

    print(f"💬 User: '{utterance}'")
    print(f"   ├─ Detected Track: [{opt_dst.current_intent.upper()}]")
    print(f"   ├─ Extracted Slots: {updated_slots}")
    print(f"   ├─ Clean Augmented Query: '{clean_query}'")

    if trigger_cutoff:
        print(f"   └─ ⚡ PROACTIVE CUTOFF TRIGGERED: \"{prompt}\"")
    else:
        final_rankings = reranker.rerank(raw_candidates, updated_slots, clean_query, top_k=5)
        top = final_rankings[0]
        print(f"   └─ 🎯 Top Ranked (#1): [{top.get('parent_asin')}] {top.get('title')[:60]}... | Final Score: {top['final_score']:.4f}")

    print("-" * 75)

=== COMPLETE PIPELINE MULTI-TURN TEST ===

💬 User: 'Looking for comfortable running shoes'
   ├─ Detected Track: [BROWSING]
   ├─ Extracted Slots: {'gender': None, 'brand': None, 'category': None, 'max_price': None, 'color': None}
   ├─ Clean Augmented Query: 'Looking for comfortable running shoes'
   └─ ⚡ PROACTIVE CUTOFF TRIGGERED: "I found many matching options! Do you have a preferred brand (e.g., Nike, ASICS) or a target price range?"
---------------------------------------------------------------------------
💬 User: 'Show me Nike shoes for men'
   ├─ Detected Track: [BUYING]
   ├─ Extracted Slots: {'gender': 'Men', 'brand': 'Nike', 'category': None, 'max_price': None, 'color': None}
   ├─ Clean Augmented Query: 'Men Nike shoes for'
   └─ 🎯 Top Ranked (#1): [B00A40GKZU] Nike Men's Air Max 90 Essential Running Shoes, Black/Black, ... | Final Score: 0.0424
---------------------------------------------------------------------------
💬 User: 'Actually, change that to ASICS instead'
   

In [30]:
import numpy as np
from typing import List, Dict, Any

def run_full_benchmark_eval(
    engine: HybridSearchEngine,
    dst: OptimizedDialogStateTracker,
    reranker: SemanticLLMReranker,
    eval_sessions: List[Dict[str, Any]],
    top_k: int = 10
):
    """
    Evaluates all 3 Hackathon Metrics:
    1. Coverage (Hit Rate@K): Catalog recall boundary capability.
    2. Precision (MRR@K): LLM reranker ability to push exact item to Rank 1.
    3. Efficiency (MTTC): Mean Turns to Conversion (fewer turns = higher efficiency score).
    """
    hits = 0
    reciprocal_ranks = []
    conversion_turns = []

    print("==========================================================")
    print(f" 📊 BENCHMARK EVALUATION ENGINE (Top-{top_k} Metrics)")
    print("==========================================================\n")

    for session_idx, session in enumerate(eval_sessions, start=1):
        dst.reset()
        target_asin = session["target_asin"]
        converted = False

        print(f"🔹 Session #{session_idx} | Target Item: [{target_asin}]")

        for turn_num, utterance in enumerate(session["turns"], start=1):
            slots = dst.update_state(utterance)
            query = dst.construct_search_query(utterance)

            candidates = engine.search(query=query, track=dst.current_intent, candidate_k=100, final_top_k=50)

            trigger_cutoff, _ = dst.should_trigger_proactive_guidance(len(candidates))
            if trigger_cutoff and turn_num == 1:
                print(f"   ├─ Turn {turn_num}: ⚡ Proactive Guidance Triggered (MTTC Efficiency Control)")
                continue

            ranked_results = reranker.rerank(candidates, slots=slots, user_query=query, top_k=top_k)
            retrieved_asins = [item.get("parent_asin", item.get("id")) for item in ranked_results]

            if target_asin in retrieved_asins:
                rank = retrieved_asins.index(target_asin) + 1
                rr = 1.0 / rank
                reciprocal_ranks.append(rr)
                hits += 1
                converted = True
                conversion_turns.append(turn_num)
                print(f"   └─ Turn {turn_num}: CONVERTED at Rank #{rank} (RR: {rr:.4f}) | Query: '{query}'")
                break
            else:
                print(f"   ├─ Turn {turn_num}: Searching... (Target not in Top-{top_k})")

        if not converted:
            reciprocal_ranks.append(0.0)
            conversion_turns.append(len(session["turns"]) + 1) # Penalty for non-conversion
            print(f"   └─ Session Failed to convert target within interaction limit.")
        print("-" * 58)

    hit_rate = (hits / len(eval_sessions)) * 100
    mrr = float(np.mean(reciprocal_ranks))
    mttc = float(np.mean(conversion_turns))

    print("\n" + "═"*58)
    print(" FINAL OFFICIAL SYSTEM METRICS SUMMARY")
    print("═"*58)
    print(f" 1. Coverage (Hit Rate@{top_k}):  {hit_rate:.2f}%  ({hits}/{len(eval_sessions)})")
    print(f" 2. Precision (MRR@{top_k}):      {mrr:.4f}")
    print(f" 3. Efficiency (MTTC):         {mttc:.2f} Turns to Conversion")
    print("═"*58 + "\n")

    return {"hit_rate": hit_rate, "mrr": mrr, "mttc": mttc}

eval_sessions = [
    {
        "target_asin": "B00A40GKZU",
        "turns": [
            "Looking for comfortable running shoes",
            "Show me Nike shoes for men"
        ]
    },
    {
        "target_asin": "B00BGX9GBY",
        "turns": [
            "Looking for comfortable running shoes",
            "Show me Nike shoes for men",
            "Actually, change that to ASICS instead"
        ]
    }
]

metrics = run_full_benchmark_eval(engine, opt_dst, reranker, eval_sessions, top_k=10)

 📊 BENCHMARK EVALUATION ENGINE (Top-10 Metrics)

🔹 Session #1 | Target Item: [B00A40GKZU]
   ├─ Turn 1: ⚡ Proactive Guidance Triggered (MTTC Efficiency Control)
   └─ Turn 2: CONVERTED at Rank #1 (RR: 1.0000) | Query: 'Men Nike shoes for'
----------------------------------------------------------
🔹 Session #2 | Target Item: [B00BGX9GBY]
   ├─ Turn 1: ⚡ Proactive Guidance Triggered (MTTC Efficiency Control)
   ├─ Turn 2: Searching... (Target not in Top-10)
   └─ Turn 3: CONVERTED at Rank #1 (RR: 1.0000) | Query: 'Men Asics'
----------------------------------------------------------

══════════════════════════════════════════════════════════
 FINAL OFFICIAL SYSTEM METRICS SUMMARY
══════════════════════════════════════════════════════════
 1. Coverage (Hit Rate@10):  100.00%  (2/2)
 2. Precision (MRR@10):      1.0000
 3. Efficiency (MTTC):         2.50 Turns to Conversion
══════════════════════════════════════════════════════════

